In [0]:
from pyspark.sql.functions import current_timestamp

from pipeline_utils import S3_BASE, cast_bronze_columns

file_list = dbutils.fs.ls(f"{S3_BASE}yellow_taxi/")
parquet_files = [f.path for f in file_list if f.path.endswith(".parquet")]

# Read each file individually and union them
dfs = [cast_bronze_columns(spark.read.parquet(f)) for f in parquet_files]
df_combined = dfs[0]
for df in dfs[1:]:
    df_combined = df_combined.unionByName(df, allowMissingColumns=True)

df_combined = df_combined.withColumn("ingestion_timestamp", current_timestamp())

df_combined.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze.taxidata.yellow_taxi_data")

In [0]:
df1 = spark.table("bronze.taxidata.yellow_taxi_data")

In [0]:
df1.printSchema()

In [0]:
from pyspark.sql.functions import current_timestamp

from pipeline_utils import S3_BASE, cast_bronze_columns

file_list = dbutils.fs.ls(f"{S3_BASE}green_taxi/")
parquet_files = [f.path for f in file_list if f.path.endswith(".parquet")]

# Read each file individually and union them
dfs = [cast_bronze_columns(spark.read.parquet(f)) for f in parquet_files]
df_combined = dfs[0]
for df in dfs[1:]:
    df_combined = df_combined.unionByName(df, allowMissingColumns=True)
    
df_combined = df_combined.withColumn("ingestion_timestamp", current_timestamp())

df_combined.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze.taxidata.green_taxi_data")

In [0]:
df2 = spark.table("bronze.taxidata.green_taxi_data")

In [0]:
df2.printSchema()